In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [6]:
PROJECT_ROOT = Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / "Raw_Dataset"
PROCESSED_DIR = PROJECT_ROOT / "Processed_Dataset"
TABLE_DIR = PROJECT_ROOT / "Outputs" / "Tables"
FIGURE_DIR = PROJECT_ROOT / "Outputs" / "Figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "diabetic_data.csv"
IDS_PATH = RAW_DIR / "IDS_mapping.csv"

print(DATA_PATH)
print(IDS_PATH)

/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Raw_Dataset/diabetic_data.csv
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Raw_Dataset/IDS_mapping.csv


## Data Preping and Cleaning

In [9]:
df_raw = pd.read_csv(DATA_PATH)
ids_mapping = pd.read_csv(IDS_PATH)

print("Raw diabetes data shape:", df_raw.shape)
print("IDS mapping shape:", ids_mapping.shape)

df_raw.head()

Raw diabetes data shape: (101766, 50)
IDS mapping shape: (67, 2)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [10]:
df_raw.columns.tolist()

['encounter_id',
 'patient_nbr',
 'race',
 'gender',
 'age',
 'weight',
 'admission_type_id',
 'discharge_disposition_id',
 'admission_source_id',
 'time_in_hospital',
 'payer_code',
 'medical_specialty',
 'num_lab_procedures',
 'num_procedures',
 'num_medications',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'diag_1',
 'diag_2',
 'diag_3',
 'number_diagnoses',
 'max_glu_serum',
 'A1Cresult',
 'metformin',
 'repaglinide',
 'nateglinide',
 'chlorpropamide',
 'glimepiride',
 'acetohexamide',
 'glipizide',
 'glyburide',
 'tolbutamide',
 'pioglitazone',
 'rosiglitazone',
 'acarbose',
 'miglitol',
 'troglitazone',
 'tolazamide',
 'examide',
 'citoglipton',
 'insulin',
 'glyburide-metformin',
 'glipizide-metformin',
 'glimepiride-pioglitazone',
 'metformin-rosiglitazone',
 'metformin-pioglitazone',
 'change',
 'diabetesMed',
 'readmitted']

In [11]:
df = df_raw.copy()

df = df.replace("?", np.nan)

missing_summary = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
    .reset_index()
)

missing_summary.columns = ["feature", "missing_percent"]
missing_summary.head(15)

,feature,missing_percent
0,weight,96.86
1,max_glu_serum,94.75
2,A1Cresult,83.28
3,medical_specialty,49.08
4,payer_code,39.56
5,race,2.23
6,diag_3,1.40
7,diag_2,0.35
8,diag_1,0.02
9,encounter_id,0.00


In [12]:
missing_summary.to_csv(TABLE_DIR / "missing_summary_raw_table1.csv", index=False)

In [13]:
row_log = []

def log_rows(step_name, data):
    row_log.append({
        "step": step_name,
        "rows": len(data),
        "columns": data.shape[1]
    })
    print(f"{step_name}: {data.shape}")
    
df_clean = df.copy()
log_rows("Raw data", df_clean)

Raw data: (101766, 50)


In [14]:
df_clean = df_clean.drop(columns=["weight", "payer_code"], errors="ignore")
log_rows("Dropped weight and payer_code", df_clean)

Dropped weight and payer_code: (101766, 48)


In [15]:
df_clean["medical_specialty"] = df_clean["medical_specialty"].fillna("Missing")
df_clean["race"] = df_clean["race"].fillna("Missing")

In [16]:
death_hospice_ids = [11, 13, 14, 19, 20, 21]

df_clean = df_clean[
    ~df_clean["discharge_disposition_id"].isin(death_hospice_ids)
].copy()

log_rows("Removed death/hospice discharge cases", df_clean)

Removed death/hospice discharge cases: (99343, 48)


In [17]:
df_clean = df_clean[df_clean["gender"] != "Unknown/Invalid"].copy()

log_rows("Removed Unknown/Invalid gender", df_clean)

Removed Unknown/Invalid gender: (99340, 48)


In [18]:
df_clean = (
    df_clean
    .sort_values(["patient_nbr", "encounter_id"])
    .drop_duplicates(subset="patient_nbr", keep="first")
    .copy()
)

log_rows("Kept first encounter per patient", df_clean)

Kept first encounter per patient: (69987, 48)


In [19]:
row_log_df = pd.DataFrame(row_log)
row_log_df

,step,rows,columns
0,Raw data,101766,50
1,Dropped weight and payer_code,101766,48
2,Removed death/hospice discharge cases,99343,48
3,Removed Unknown/Invalid gender,99340,48
4,Kept first encounter per patient,69987,48


In [20]:
row_log_df.to_csv(TABLE_DIR / "preprocessing_row_log.csv", index=False)

In [21]:
df_clean["readmitted_30"] = (df_clean["readmitted"] == "<30").astype(int)

df_clean["readmitted_30"].value_counts()

readmitted_30
0    63702
1     6285
Name: count, dtype: int64

In [22]:
readmission_rate = df_clean["readmitted_30"].mean() * 100
print(f"30-day readmission rate: {readmission_rate:.2f}%")

30-day readmission rate: 8.98%


## Create Hb1Ac Group

In [23]:
def make_hba1c_group(row):
    a1c = row["A1Cresult"]
    change = row["change"]

    if a1c == "None":
        return "No test"
    elif a1c == ">8" and change == "Ch":
        return "High, med changed"
    elif a1c == ">8" and change == "No":
        return "High, med not changed"
    else:
        return "Normal"

df_clean["hba1c_group"] = df_clean.apply(make_hba1c_group, axis=1)

df_clean["hba1c_group"].value_counts()

hba1c_group
Normal                   63748
High, med changed         4058
High, med not changed     2181
Name: count, dtype: int64

In [24]:
def make_primary_diagnosis_group(code):
    if pd.isna(code):
        return "Other"

    code = str(code)

    if code.startswith("V") or code.startswith("E"):
        return "Other"

    try:
        code_num = float(code)
    except ValueError:
        return "Other"

    if (390 <= code_num <= 459) or code_num == 785:
        return "Circulatory"
    elif (460 <= code_num <= 519) or code_num == 786:
        return "Respiratory"
    elif (520 <= code_num <= 579) or code_num == 787:
        return "Digestive"
    elif int(code_num) == 250:
        return "Diabetes"
    elif 800 <= code_num <= 999:
        return "Injury"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal"
    elif (580 <= code_num <= 629) or code_num == 788:
        return "Genitourinary"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    else:
        return "Other"

df_clean["primary_diagnosis"] = df_clean["diag_1"].apply(make_primary_diagnosis_group)

df_clean["primary_diagnosis"].value_counts()

primary_diagnosis
Circulatory        21389
Other              12134
Respiratory         9491
Digestive           6488
Diabetes            5748
Injury              4694
Musculoskeletal     4064
Genitourinary       3441
Neoplasms           2538
Name: count, dtype: int64

In [25]:
age_group_map = {
    "[0-10)": "<=30",
    "[10-20)": "<=30",
    "[20-30)": "<=30",
    "[30-40)": "30-60",
    "[40-50)": "30-60",
    "[50-60)": "30-60",
    "[60-70)": ">60",
    "[70-80)": ">60",
    "[80-90)": ">60",
    "[90-100)": ">60"
}

df_clean["age_group"] = df_clean["age"].map(age_group_map)

df_clean["age_group"].value_counts()

age_group
>60      46308
30-60    21871
<=30      1808
Name: count, dtype: int64

In [31]:
def make_discharge_group(x):
    if x == 1:
        return "Home"
    else:
        return "Other"

df_clean["discharge_group"] = df_clean["discharge_disposition_id"].apply(make_discharge_group)

def make_race_group(x):
    if x in ["AfricanAmerican", "Caucasian", "Missing"]:
        return x
    else:
        return "Other"

df_clean["race_group"] = df_clean["race"].apply(make_race_group)

def make_admission_source_group(x):
    if x == 7:
        return "Emergency room"
    elif x == 1:
        return "Physician/clinic referral"
    else:
        return "Other"

df_clean["admission_source_group"] = df_clean["admission_source_id"].apply(make_admission_source_group)

def make_medical_specialty_group(x):
    if x == "InternalMedicine":
        return "Internal Medicine"
    elif x == "Cardiology":
        return "Cardiology"
    elif x in ["Surgery-General", "Surgery-Cardiovascular/Thoracic", 
               "Surgery-Neuro", "Surgery-Vascular", "Surgery-Plastic",
               "Surgery-Cardiovascular", "Surgery-Colon&Rectal",
               "Surgery-Pediatric", "Surgery-Maxillofacial",
               "Surgery-PlasticwithinHeadandNeck"]:
        return "Surgery"
    elif x in ["Family/GeneralPractice"]:
        return "Family/General Practice"
    elif x == "Missing":
        return "Missing"
    else:
        return "Other"

df_clean["medical_specialty_group"] = df_clean["medical_specialty"].apply(make_medical_specialty_group)

df_clean["medical_specialty_group"].value_counts()

medical_specialty_group
Missing                    33652
Other                      12915
Internal Medicine          10641
Family/General Practice     4978
Cardiology                  4207
Surgery                     3594
Name: count, dtype: int64

In [32]:
def table3_summary(data, column):
    summary = (
        data.groupby(column, dropna=False)
        .agg(
            encounters=("readmitted_30", "size"),
            readmitted=("readmitted_30", "sum"),
            readmission_rate=("readmitted_30", "mean")
        )
        .reset_index()
    )

    summary["population_percent"] = summary["encounters"] / len(data) * 100
    summary["readmission_rate"] = summary["readmission_rate"] * 100

    summary = summary[
        [column, "encounters", "population_percent", "readmitted", "readmission_rate"]
    ]

    summary["population_percent"] = summary["population_percent"].round(1)
    summary["readmission_rate"] = summary["readmission_rate"].round(1)

    return summary

In [33]:
hba1c_summary = table3_summary(df_clean, "hba1c_group")
hba1c_summary
hba1c_summary.to_csv(TABLE_DIR / "table3_hba1c_summary.csv", index=False)

In [34]:
gender_summary = table3_summary(df_clean, "gender")
discharge_summary = table3_summary(df_clean, "discharge_group")
admission_summary = table3_summary(df_clean, "admission_source_group")
specialty_summary = table3_summary(df_clean, "medical_specialty_group")
diagnosis_summary = table3_summary(df_clean, "primary_diagnosis")
race_summary = table3_summary(df_clean, "race_group")
age_summary = table3_summary(df_clean, "age_group")

display(gender_summary)
display(discharge_summary)
display(admission_summary)
display(specialty_summary)
display(diagnosis_summary)
display(race_summary)
display(age_summary)

,gender,encounters,population_percent,readmitted,readmission_rate
0,Female,37239,53.2,3365,9.0
1,Male,32748,46.8,2920,8.9


,discharge_group,encounters,population_percent,readmitted,readmission_rate
0,Home,44320,63.3,3079,6.9
1,Other,25667,36.7,3206,12.5


,admission_source_group,encounters,population_percent,readmitted,readmission_rate
0,Emergency room,37271,53.3,3452,9.3
1,Other,10968,15.7,960,8.8
2,Physician/clinic referral,21748,31.1,1873,8.6


,medical_specialty_group,encounters,population_percent,readmitted,readmission_rate
0,Cardiology,4207,6.0,303,7.2
1,Family/General Practice,4978,7.1,485,9.7
2,Internal Medicine,10641,15.2,1039,9.8
3,Missing,33652,48.1,3110,9.2
4,Other,12915,18.5,1062,8.2
5,Surgery,3594,5.1,286,8.0


,primary_diagnosis,encounters,population_percent,readmitted,readmission_rate
0,Circulatory,21389,30.6,2070,9.7
1,Diabetes,5748,8.2,524,9.1
2,Digestive,6488,9.3,520,8.0
3,Genitourinary,3441,4.9,309,9.0
4,Injury,4694,6.7,507,10.8
5,Musculoskeletal,4064,5.8,341,8.4
6,Neoplasms,2538,3.6,230,9.1
7,Other,12134,17.3,1091,9.0
8,Respiratory,9491,13.6,693,7.3


,race_group,encounters,population_percent,readmitted,readmission_rate
0,AfricanAmerican,12627,18.0,1095,8.7
1,Caucasian,52305,74.7,4807,9.2
2,Missing,1917,2.7,141,7.4
3,Other,3138,4.5,242,7.7


,age_group,encounters,population_percent,readmitted,readmission_rate
0,30-60,21871,31.3,1574,7.2
1,<=30,1808,2.6,112,6.2
2,>60,46308,66.2,4599,9.9


In [35]:
gender_summary.to_csv(TABLE_DIR / "table3_gender_summary.csv", index=False)
discharge_summary.to_csv(TABLE_DIR / "table3_discharge_summary.csv", index=False)
admission_summary.to_csv(TABLE_DIR / "table3_admission_summary.csv", index=False)
specialty_summary.to_csv(TABLE_DIR / "table3_specialty_summary.csv", index=False)
diagnosis_summary.to_csv(TABLE_DIR / "table3_diagnosis_summary.csv", index=False)
race_summary.to_csv(TABLE_DIR / "table3_race_summary.csv", index=False)
age_summary.to_csv(TABLE_DIR / "table3_age_summary.csv", index=False)

In [36]:
df_clean.to_csv(PROCESSED_DIR / "diabetic_data_cleaned_stage1.csv", index=False)

print("Saved cleaned dataset to:")
print(PROCESSED_DIR / "diabetic_data_cleaned_stage1.csv")

Saved cleaned dataset to:
/Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Processed_Dataset/diabetic_data_cleaned_stage1.csv
